# Data Cleaning

In [6]:
import pandas as pd
import geopandas as gpd 

In [7]:
# Read the data file
reports = pd.read_csv("../data/raw/stzh.zwn_meldungen_p.csv")

# Date transformation
reports["requested_datetime"] = pd.to_datetime(reports["requested_datetime"])

# Data from the last 10 years (2016-2025)
start_date = "2016-01-01"
end_date = "2025-12-31"
reports = reports[(reports["requested_datetime"] >= start_date) & 
                  (reports["requested_datetime"] <= end_date)]


# Date and year column
reports["date"] = reports["requested_datetime"].dt.date
reports["year"] = reports["requested_datetime"].dt.year

# Extract useful columns
print(reports.columns)
reports = reports[["date","service_code", "e", "n", "geometry"]]

display(reports.sort_values(by="date", ascending=False).head())
display(reports.sort_values(by="date", ascending=True).head())

# Save the cleand .csv in the "processed" data folder
reports.to_csv("../data/processed/reports_cleaned.csv", index=False)

Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry',
       'date', 'year'],
      dtype='object')


,date,service_code,e,n,geometry
69250,2025-12-30,Abfall/Sammelstelle,2680119,1250528,POINT (2680119 1250528)
69240,2025-12-30,Abfall/Sammelstelle,2683159,1252891,POINT (2683159 1252891)
69230,2025-12-30,Abfall/Sammelstelle,2683762,1252276,POINT (2683762 1252276)
69231,2025-12-30,Grünflächen/Spielplätze,2682397,1247535,POINT (2682397 1247535)
69232,2025-12-30,Abfall/Sammelstelle,2678939,1252705,POINT (2678939 1252705)


,date,service_code,e,n,geometry
7016,2016-01-01,Abfall/Sammelstelle,2679329,1248930,POINT (2679329 1248930)
7017,2016-01-02,Abfall/Sammelstelle,2683472,1251981,POINT (2683472 1251981)
7018,2016-01-02,Abfall/Sammelstelle,2682074,1248879,POINT (2682074 1248879)
7020,2016-01-03,Beleuchtung/Uhren,2684777,1251971,POINT (2684777 1251971)
7021,2016-01-03,Signalisation/Lichtsignal,2680937,1247118,POINT (2680937 1247118)


In [8]:
# Lakes layer auf Stadt Zürich zuschneiden
# Read the data file
zh_quartiere = gpd.read_file("../data/raw/seen_stadtZH.gpkg")

lakes = gpd.read_file("../data/raw/AV_Gewasser_-OGD.gpkg")
lakes = lakes.to_crs(epsg=2056)

perimeter = zh_quartiere.union_all()

lakes_zh = gpd.clip(lakes, perimeter)

zürichsee_gdf = lakes_zh[lakes_zh["gewaessername"] == "Zürichsee"]

zürichsee_gdf.to_file("../data/processed/zurichsee.gpkg", driver="GPKG")